## First

In [1]:
import os
from lm_studio_code import qwen
import json
import tqdm
from comfy_code import app
import time
from upload_to_shopify import db, shopify
import shutil
import sys, threading
from PyQt5.QtWidgets import QApplication
from upload_to_shopify import product_editor as pe

# Root directory where this script is located
ROOT_DIR = os.getcwd()

DOWNLOADS_DIR = os.path.join(ROOT_DIR, "downloads", "images")
ORIGINAL_IMAGES_DIR = os.path.join(ROOT_DIR, "original_images")
GENERATED_IMAGES_DIR = os.path.join(ROOT_DIR, "generated_images")
COMFY_OUTPUT_IMAGES_DIR = os.path.join(ROOT_DIR, "ComfyUI_windows_portable", "ComfyUI","output")

print("DOWNLOADS_DIR:", DOWNLOADS_DIR)
print("ORIGINAL_IMAGES_DIR:", ORIGINAL_IMAGES_DIR)
print("GENERATED_IMAGES_DIR:", GENERATED_IMAGES_DIR)
print("COMFY_OUTPUT_IMAGES_DIR:", COMFY_OUTPUT_IMAGES_DIR)

ALL_IMAGES = []
for file in os.listdir(DOWNLOADS_DIR):
    if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp")):
        ALL_IMAGES.append(os.path.join(DOWNLOADS_DIR, file))
app.start_comfyui()
products_db = db.ProductsDB()
sp = shopify.ShopifyUploader(products_db)

DOWNLOADS_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\downloads\images
ORIGINAL_IMAGES_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\original_images
GENERATED_IMAGES_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\generated_images
COMFY_OUTPUT_IMAGES_DIR: c:\Users\FIRST\Desktop\piyagushopify\llmautomation\ComfyUI_windows_portable\ComfyUI\output
Starting ComfyUI...
COMFY_ROOT: C:\Users\FIRST\Desktop\piyagushopify\llmautomation\ComfyUI_windows_portable
ComfyUI is ready
Authenticated with Shopify at 2026-06-18 17:07:45.236963
Fetched 21 collections from Shopify at 2026-06-18 17:07:45.671794
fetched publications : ['gid://shopify/Publication/171978195083', 'gid://shopify/Publication/171978260619', 'gid://shopify/Publication/172730908811', 'gid://shopify/Publication/175160459403']


In [2]:
qwen.load_model_if_needed()
products = []
for image_path in tqdm.tqdm(ALL_IMAGES[:5]):
    response = qwen.generate_response(image_path)
    content = next(
        x["content"]
        for x in response["output"]
        if x["type"] == "message"
    )
    product = json.loads(content)
    product["image_path"] = image_path
    products.append(product)
qwen.unload_model()

Loading model: qwen3.5-35b-a3b


100%|██████████| 5/5 [18:09<00:00, 217.90s/it]


{'instance_id': 'qwen3.5-35b-a3b'}

In [ ]:
workflow = app.load_workflow()
comfy_prompts = app.load_prompts()
for i, product in enumerate(tqdm.tqdm(products)):
    generated_image_prefix = f"generated_{int(time.time() * 1000)}_{i}_"

    prompt = comfy_prompts[product["product_type"]].replace(
        "{set_includes}",
        ", ".join(product["set_includes"])
    )

    app.run_workflow(
        image_path=product["image_path"],
        prompt=prompt,
        workflow=workflow,
        output_prefix=generated_image_prefix
    )

    product["GENERATED_IMAGE_LOCATION"] = os.path.join(COMFY_OUTPUT_IMAGES_DIR,generated_image_prefix+"_00001_.png")
app.free_resources()

  0%|          | 0/5 [00:00<?, ?it/s]

Prompt ID: b993d48b-3739-4680-97ce-04c03cf87a6e


 20%|██        | 1/5 [02:46<11:07, 166.76s/it]

Generation completed
Prompt ID: f696bb6e-bc85-441e-86fd-ff3031b5a40a


 40%|████      | 2/5 [05:49<08:48, 176.31s/it]

...

 80%|████████  | 4/5 [11:49<02:58, 178.54s/it]

Generation completed
Prompt ID: 328c9faa-e7f1-4fbc-958e-9a473532da2f


100%|██████████| 5/5 [14:57<00:00, 179.50s/it]

Generation completed


<Response [200]>

In [ ]:
for product in products:
    timestamp_ms = int(time.time() * 1000)
    product['title'] = f"{product['title']} | PiyaGuJwells"
    product['subtitle'] = f"{product['subtitle']} | PiyaGuJwells"
    product['image_path'] = shutil.move(product['image_path'], ORIGINAL_IMAGES_DIR)
    product['GENERATED_IMAGE_LOCATION'] = shutil.move(product['GENERATED_IMAGE_LOCATION'], GENERATED_IMAGES_DIR)
    product["price"] = product["price"]*1.8
    products_db.add_product(product, timestamp_ms, product['image_path'], product['GENERATED_IMAGE_LOCATION'])

## UI

In [2]:

sys.path.insert(0, 'upload_to_shopify')  # folder containing product_editor.py and db.py

def run():
    app = QApplication.instance() or QApplication(sys.argv)
    app.setStyle('Fusion')
    win = pe.ProductEditorWindow(db_path='products.db')
    win.show()
    app.exec_()

threading.Thread(target=run, daemon=True).start()

## Upload

In [7]:
def check_errors(results):
    for k,v in results.items():
        if v is None or k == "shopify_image_url":
            continue
        for k2,v2 in v["data"].items():
            if v2["userErrors"]:
                print(v["data"])
                return 1
    return 0
while True:
    p = products_db.get_next_ready_for_upload_product()
    if p is None:
        break
    if p["price"] <500:
        print("check_price for ",p)
        break
    results = sp.upload_product(p)
    if check_errors(results):
        break
    products_db.mark_uploaded(p["id"], results['shopify_image_url'], results["product_upload_response"]["data"]["productCreate"]["product"]["id"])


check_price for  {'id': 4, 'sku': 1781779082469, 'title': 'Elegant Gold Tone Stainless Steel Watch Bracelet with Crystal Accents | PiyaGuJwells', 'subtitle': 'Stylish Timepiece for Daily and Office Wear | PiyaGuJwells', 'meta_title': 'Gold Tone Watch Bracelet | Imitation Jewelry', 'meta_description': 'Shop this elegant gold tone watch bracelet with crystal accents. Stainless steel construction suitable for daily wear and office time. Available in sizes 2.5 & 2.6.', 'alt_text': 'Gold imitation watch bracelet with white dial and crystals', 'variants': [{'type': 'Size', 'values': ['2.5', '2.6']}], 'product_type': 'Bracelets', 'product_tags': ['office-time', 'daily-wear', 'office-daily', 'under-999'], 'set_includes': ['Watch Bracelet'], 'price': 0.0, 'original_image_path': 'c:\\Users\\FIRST\\Desktop\\piyagushopify\\llmautomation\\original_images\\IMG-20260204-WA0258.jpg', 'generated_image_path': 'c:\\Users\\FIRST\\Desktop\\piyagushopify\\llmautomation\\generated_images\\generated_178178269

## Regenerate

In [8]:
products_left = products_db.get_regenerate_image_products()
workflow = app.load_workflow()
comfy_prompts = app.load_prompts()
for i, product in enumerate(tqdm.tqdm(products_left)):
    generated_image_prefix = f"generated_{int(time.time() * 1000)}_{i}_"

    prompt = comfy_prompts["Alternative"].replace(
        "{set_includes}",
        ", ".join(product["set_includes"])
    )

    app.run_workflow(
        image_path=product["original_image_path"],
        prompt=prompt,
        workflow=workflow,
        output_prefix=generated_image_prefix
    )

    if os.path.exists(product["generated_image_path"]):
        os.remove(product["generated_image_path"])

    GENERATED_IMAGE_LOCATION = os.path.join(COMFY_OUTPUT_IMAGES_DIR,generated_image_prefix+"_00001_.png")
    GENERATED_IMAGE_LOCATION = shutil.move(GENERATED_IMAGE_LOCATION, GENERATED_IMAGES_DIR)
    products_db.update_product_generated_image_path(product["id"],GENERATED_IMAGE_LOCATION)
    products_db.mark_for_regeneration(product["id"],False)
app.free_resources()


0it [00:00, ?it/s]


<Response [200]>